# 18 — Trigger a CellReg run from the database

**Edit cell 1. Run the rest in order.** Everything before step 3 is read-only.

This registers cells across the sessions of one *site* (one field of view over days). The
point of doing it from the database is that the **curation of every column is declared
before anything runs**, so the provenance is a recorded fact. Leon's historical exports
copied `<ScanPath>/suite2p/plane0/Fall.mat` — a path with no curation in it — which is why
854 of the 1,266 historical columns had to be attached afterwards by content matching or
convention instead of simply being read off.

Notebook 17 is the other direction: it parses registrations that already exist.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  THE ONLY CELL YOU EDIT
# ─────────────────────────────────────────────────────────────────────────────
SUBJECT   = 'ROS-1881'          # animal
SITE      = 'sess9FPXNI8U'      # same_site_id = the site's FIRST session id
LABEL     = 'Gratings'          # free text, goes in the folder name and the task

STIMULUS  = 'Moving_Gratings'   # event-type prefix a scan must carry; None = any scan
PARAMSET  = 10                  # suite2p paramset_idx
CURATION  = 'max'               # 'max' = newest curation of each scan, or an int
#  Defaults above are a site that really does return columns, so step 1 shows
#  something on a first run. Two worked variants:
#    ROS-2101 / sess9FUCURTR with STIMULUS=None -> the mixed-zoom case: 6 scans,
#       4 kept at 466 um and 2 dropped at 932 um (its events are Bonsai VR
#       strings, so the Moving_Gratings filter would return nothing)
#    SCANS=['scan...', ...] -> bypass the stimulus filter entirely
SCANS     = None                # None = every scan at the site matching STIMULUS

RUN_WRITES = False              # <<< True actually declares the task and writes files

In [ ]:
import os
import sys
import pathlib
import datajoint as dj

# ── Which `adamacs` am I importing? ──────────────────────────────────────────
# `adamacs` is installed in development mode in several lab environments pointing at
# ~/adamacs, which does NOT contain the cell_registration code. Importing then fails with
# "cannot import name 'cellreg_plan' from 'adamacs.ingest'" naming that other checkout, and
# the message gives no hint that the wrong repository is on the path. Put THIS repository
# first, then check what actually got imported.
_HERE = pathlib.Path.cwd()
ROOT = next((p for p in [_HERE, *_HERE.parents]
             if (p / 'adamacs' / 'ingest' / 'cellreg_plan.py').exists()),
            pathlib.Path(os.path.expanduser('~/adamacs_ingest')))
sys.path.insert(0, str(ROOT))

# ── Credentials and the schema prefix ───────────────────────────────────────
# dj_local_conf.json is NOT in the same place on every host: tatchu3 has it only at
# ~/adamacs/dj_local_conf.json, ibehavegpu1 also has one in the repo root. Load it BEFORE
# importing anything from adamacs -- the prefix is read at import, and without credentials
# DataJoint prompts for a username on stdin.
for _cand in (os.environ.get('DJ_LOCAL_CONF'),
              os.path.expanduser('~/adamacs/dj_local_conf.json'),
              str(ROOT / 'dj_local_conf.json'),
              'dj_local_conf.json'):
    if _cand and os.path.exists(_cand):
        dj.config.load(_cand); print('dj config:', _cand); break
else:
    raise FileNotFoundError('no dj_local_conf.json found; set DJ_LOCAL_CONF')

import adamacs
if not str(pathlib.Path(adamacs.__file__).resolve()).startswith(str(ROOT.resolve())):
    raise ImportError(
        f"'adamacs' resolved to {adamacs.__file__}\ninstead of {ROOT}. Another checkout is "
        f"installed in this environment. RESTART THE KERNEL and run this cell first -- "
        f"sys.path cannot displace an already-imported package.")
print('adamacs package:', os.path.dirname(adamacs.__file__))

from adamacs.ingest import cellreg_plan as plan_mod

conn = dj.conn()
print('database prefix:', dj.config['custom']['database.prefix'])

## 1. See what would be registered

`plan_site_task` does the two things that used to be done wrong by hand:

- it groups the site's scans by **field of view** and keeps only the largest consistent
  group, because CellReg takes one `microns_per_pixel` for the whole run and never
  rescales — ROS-2101 has four scans at 466 µm and two at 932 µm over the same 512 px;
- it picks the **curation** by an explicit rule and reports how many each scan has.

It identifies the stimulus from `event.Event` (`Moving_Gratings:<direction>`), because
`scan.Scan.scan_notes` is free text and is often literally `'none'`.

Nothing is written. Read the excluded list — that is where the surprises are.

In [ ]:
plan = plan_mod.plan_site_task(
    conn, subject=SUBJECT, site=SITE, paramset_idx=PARAMSET,
    curation=CURATION, stimulus=STIMULUS, scans=SCANS)

plan_mod.print_plan(plan)

## 2. Settings

Transcribed into the driver's vocabulary, not the schema's. A paramset recovered from a log
describes a past run and **cannot drive a new one**: it carries CellReg's derived `n_bins`
and `initial_threshold`, which `settings_payload` refuses by design.

`reference_session_index` is 1-based and defaults to the middle column.

In [ ]:
SETTINGS = {
    'input_format': 'Suite2p',
    'alignment_type': 'Translations and Rotations',
    'initial_registration_type': 'Spatial correlation',
    'registration_approach': 'Probabilistic',
    'model_type': 'Spatial correlation',        # singular -- the driver uses strcmp
    'maximal_distance': 12.0,
    'p_same_threshold': 0.3,
    'microns_per_pixel': plan['microns_per_pixel'],
    'reference_session_index': max(1, len(plan['columns']) // 2),
}

from adamacs.ingest import cellreg_export as EX
for k, v in EX.settings_payload(SETTINGS).items():
    print(f'   {k}: {v!r}')

## 3. Declare the task  `# WRITES`

This is the step that records the manifest. Set `RUN_WRITES = True` in cell 1 first.

It inserts the site, registers the paramset, and writes one `CellRegTask.Session` row per
column naming its curation. Then `CellRegExport` builds the input files — one
`Fall_<NN>_<scan>.mat` per column plus `settings.mat` — into a **new** directory. It can
never touch a suite2p tree: `assert_safe_target` refuses any path inside one, any file
named `Fall.mat`, and any overwrite.

In [ ]:
if not RUN_WRITES:
    print('RUN_WRITES is False -- nothing written. Set it True in cell 1 to proceed.')
else:
    import datetime
    from adamacs.pipeline import cell_registration as cr   # importing the pipeline WRITES
                                                           # a Lab row; never do it on a
                                                           # read-only connection
    site_key = {'subject': SUBJECT, 'same_site_id': SITE}
    cr.RegistrationSite.insert1(
        {**site_key, 'site_description': f'{SUBJECT} {LABEL} site',
         'site_id_source': 'same_site_id', 'created': datetime.datetime.now()},
        skip_duplicates=True)

    pset = cr.CellRegParamSet.register(
        f'prob_tr_p{SETTINGS["p_same_threshold"]}_mpp{SETTINGS["microns_per_pixel"]}'
        f'_{cr.config_hash(SETTINGS)[:8]}'[:40],
        SETTINGS, matching_method='cellreg_probabilistic',
        label=f'driver settings for {SUBJECT} {LABEL}')

    out_dir = EX.export_dir_for(site_key, label=LABEL.replace(' ', '_'), task_id=0)
    os.makedirs(out_dir.parent, exist_ok=True)

    task_key = cr.CellRegTask.auto_generate_entries(
        site_key, plan_mod.curation_keys(plan), paramset=pset, task_mode='trigger',
        output_dir=str(out_dir), label=LABEL,
        note=f'{len(plan["columns"])} columns, curation rule {CURATION!r}, '
             f'stimulus {STIMULUS}; {len(plan["excluded"])} scans excluded',
        created_by=os.environ.get('USER', ''))
    print('task:', task_key)

    cr.CellRegExport.populate(task_key, suppress_errors=False)
    exp = (cr.CellRegExport & task_key).fetch1()
    print(f'exported {exp["n_files"]} files, {exp["n_masks_total"]:,} masks')
    print('directory:', exp['export_dir'])
    for f in (cr.CellRegExport.File & task_key).fetch(order_by='column_index', as_dict=True):
        print(f'   {f["column_index"]}  {f["file_name"]:<34} cur{f["curation_id"]} '
              f'{f["n_masks"]:>6,} masks')

## 4. Run CellReg  `# on ibehavegpu1 only`

MATLAB lives under `backup_user` on ibehavegpu1 and must not run on tatchu3 — a CellReg run
exhausted its RAM and took the database server down once. The worker checks the host,
MATLAB and the toolbox at startup and refuses rather than dying half way.

**Dry run is the default. `--execute` is what writes.** There is no `--dry-run` flag.

In [ ]:
print(f'''ssh ibehavegpu1
sudo -u backup_user -i
cd ~/adamacs_ingest
./scripts/cellreg_worker_ibehavegpu.sh --once            # reports, writes nothing
./scripts/cellreg_worker_ibehavegpu.sh --execute --once   # runs MATLAB, registers, ingests
''')

The worker does four things per pass, and stage 2b is the one that used to be missing:

1. `CellRegExport.populate` — build inputs for any `trigger` task
2. MATLAB `cellreg_run.m` over directories that have inputs but no `Results/`
3. **`register_results`** — create the `CellRegRun` and read the manifest, recording
   `resolution_method='manifest'`. Without it `CellRegResult.populate` has nothing to do,
   because it is keyed on `CellRegRun`: the worker ran registrations and ingested none of
   them while reporting success.
4. `CellRegResult.populate` + `MaskCoordinates.populate`

Progress is posted to Slack every 30 minutes by `notify_cellreg.py` (SOPs repo, cron
`5,35`), including the site, every scan and its curation id.

## 5. Check what landed

The declaration is the test: every column should resolve as `manifest`, and the mask count
per scan should equal the declared curation's own count. A centroid count that disagrees
raises rather than silently demoting to a weaker method.

In [ ]:
import collections
from adamacs.pipeline import cell_registration as cr

runs = (cr.CellRegRun & f"result_file LIKE '%{SITE}%'").fetch(as_dict=True)
for run in runs:
    rid = run['cellreg_run_id']
    cols = (cr.CellRegRun.Session & {'cellreg_run_id': rid}).fetch(
        order_by='column_index', as_dict=True)
    print(f"run {rid[:12]} · {run['n_sessions']} columns · "
          f"{run['n_registered_cells']:,} cells · {run['run_note']}")
    print('   methods:', dict(collections.Counter(c['resolution_method'] for c in cols)),
          '| unresolved:',
          len(cr.CellRegRun.UnresolvedColumn & {'cellreg_run_id': rid}))
    per_scan = collections.Counter(
        (cr.CellRegResult.Member & {'cellreg_run_id': rid}).fetch('scan_id'))
    for c in cols:
        declared = len(cr.CellRegTask.Session
                       & {k: c[k] for k in ('session_id', 'scan_id', 'paramset_idx',
                                            'curation_id')})
        print(f"      {c['scan_id']} cur{c['curation_id']}: "
              f"{per_scan.get(c['scan_id'], 0):>6,} members  "
              f"{'declared' if declared else 'NOT in any task manifest'}")

## 6. What this never does

`Segmentation.mask.roi_group` is **not written by any step above**. Cross-session identity
lives in `CellRegResult.Cell` / `.Member`; `roi_group` stays a derived projection in
`RoiGroupProjection`, which carries the `cell_index` that explains each value next to the
run that produced it. Nothing here calls it.

That matters because the legacy path still overwrites `roi_group` in place: between
2026-10-20 and 2026-10-23 the non-zero count on pre-cutoff curations moved from 1,041,040
to 1,070,826 without any run of this pipeline.

---

**See also** — notebook 17 (parsing existing registrations, and a worked
longitudinal-tuning example) · `adamacs/ingest/cellreg_plan.py` (the planner) ·
`Database/Schemas/roselab_cell_registration.md` in the wiki.